# data preprocessing
It is a code implementation for data preprocessing to derive those columns the model training needs. The preprocessing contains three parts: interpolating the points with the time interval of five minutes, deriving the distance between current positions and the nearest coastline, and derive the Next Key Point (NKP) based on the priciples we set. Although we will not publish the training dataset, it is a sample for data preprocessing. 

In [ ]:
import pandas as pd
import numpy as np
from math import radians, cos, sin, atan2, sqrt, degrees, ceil
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path

root = Path("/path/to/") # [TODO] the root path of dataset
file_path = root / "dataset.xlsx"
dataset = pd.read_excel(file_path)
dataset['postime'] = pd.to_datetime(dataset['postime'])
data_use = dataset[['mmsi', 'postime', 'lat', 'lon', 'sog', 'cog', 'vessel_sub2_type', 'quarter', "leg_end_port_code", "目的港经度", "目的港纬度"]]

def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    r = 6371000
    return c * r

def calculate_bearing(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    x = cos(lat2) * sin(dlon)
    y = cos(lat1) * sin(lat2) - sin(lat1) * cos(lat2) * cos(dlon)
    bearing = atan2(x, y)
    bearing = degrees(bearing)
    assert bearing == bearing, f"bearing has NaN: lon1={lon1}, lat1={lat1}, lon2={lon2}, lat2={lat2}, x={x}, y={y}"
    return (bearing + 360) % 360

all_ships_data = []

for mmsi, group in tqdm(data_use.groupby('mmsi')):
    group = group.sort_values('postime').reset_index(drop=True)
    new_start = group['postime'].iloc[0]
    new_end = group['postime'].iloc[-1]
    diff = new_end - new_start
    diff_5_minutes = diff.total_seconds() / 300
    len_df = ceil(diff_5_minutes) - 1
    print(f"processing vessel {mmsi}, contains {len_df} lines in total")

    times_range = pd.date_range(start=new_start, periods=len_df, freq='5min')
    new_df = pd.DataFrame({'mmsi': mmsi, 'postime': times_range})
    new_df[['lat', 'lon', 'sog', 'cog']] = np.nan
    new_df['vessel_sub2_type'] = group['vessel_sub2_type'].iloc[0]
    new_df['quarter'] = np.nan

    for i, row in tqdm(new_df.iterrows(), desc="\t"):
        time_diff = (group['postime'] - row['postime']).abs()
        close_points = time_diff[time_diff <= pd.Timedelta(seconds=5)]
        if not close_points.empty:
            idx = close_points.idxmin()
            for col in ['lat', 'lon', 'sog', 'cog', 'quarter', "leg_end_port_code"]:
                new_df.loc[i, col] = group.loc[idx, col]
            new_df.loc[i, "target_lat"] = group.loc[idx, "目的港纬度"]
            new_df.loc[i, "target_lon"] = group.loc[idx, "目的港经度"]

    # interpolation
    interp_indices = new_df[new_df['lat'].isna()].index
    group_times = group['postime'].astype(np.int64) // 10**9

    for i in tqdm(interp_indices, desc="\t"):
        current_time = new_df.loc[i, 'postime'].to_pydatetime()
        current_ts = current_time.timestamp()

        prev_mask = group['postime'] < current_time
        next_mask = group['postime'] > current_time

        if prev_mask.any():
            prev_idx = prev_mask[prev_mask].index.max()
            prev_time = group.loc[prev_idx, 'postime'].to_pydatetime()
            prev_ts = prev_time.timestamp()
        else:
            prev_idx = None

        if next_mask.any():
            next_idx = next_mask[next_mask].index.min()
            next_time = group.loc[next_idx, 'postime'].to_pydatetime()
            next_ts = next_time.timestamp()
        else:
            next_idx = None

        if prev_idx is not None and next_idx is not None:
            total_time = next_ts - prev_ts
            time_ratio = (current_ts - prev_ts) / total_time

            prev_lat, prev_lon = group.loc[prev_idx, ['lat', 'lon']]
            next_lat, next_lon = group.loc[next_idx, ['lat', 'lon']]
            new_lat = prev_lat + (next_lat - prev_lat) * time_ratio
            new_lon = prev_lon + (next_lon - prev_lon) * time_ratio

            distance = haversine(prev_lon, prev_lat, next_lon, next_lat)
            time_diff_hours = total_time / 3600
            sog_knots = (distance / 1852) / time_diff_hours
            cog = calculate_bearing(prev_lon, prev_lat, next_lon, next_lat)

            new_df.loc[i, 'lat'] = new_lat
            new_df.loc[i, 'lon'] = new_lon
            new_df.loc[i, 'sog'] = sog_knots
            new_df.loc[i, 'cog'] = cog
            new_df.loc[i, "leg_end_port_code"] = group.loc[next_idx, "leg_end_port_code"]
            new_df.loc[i, "target_lat"] = group.loc[next_idx, "目的港纬度"]
            new_df.loc[i, "target_lon"] = group.loc[next_idx, "目的港经度"]

            prev_q = group.loc[prev_idx, 'quarter']
            next_q = group.loc[next_idx, 'quarter']
            q_interp = round(prev_q + (next_q - prev_q) * time_ratio)
            q_interp = max(1, min(4, int(q_interp)))
            new_df.loc[i, 'quarter'] = q_interp

    new_df['quarter'] = new_df['quarter'].astype(int)
    all_ships_data.append(new_df)

# combine all the vessel data
result_df = pd.concat(all_ships_data, ignore_index=True)

# output to CSV
output_dir = root / 'dataset_regular.csv'
result_df.to_csv(output_dir, index=False)
print(f"✅ the data have saved to '{output_dir}.csv'")

  0%|          | 0/4 [00:00<?, ?it/s]

正在处理船舶215347000,共包括130558条数据


	: 130558it [01:59, 1089.13it/s]
 25%|██▌       | 1/4 [05:59<17:58, 359.35s/it]

正在处理船舶218400000,共包括127331条数据


	: 127331it [01:54, 1107.56it/s]
 50%|█████     | 2/4 [11:45<11:43, 351.75s/it]

正在处理船舶563199600,共包括130935条数据


	: 130935it [01:53, 1154.32it/s]
 75%|███████▌  | 3/4 [17:34<05:50, 350.46s/it]

正在处理船舶563219100,共包括127289条数据


	: 127289it [02:00, 1053.26it/s]
100%|██████████| 4/4 [23:29<00:00, 352.38s/it]


✅ 已输出所有船舶的均匀时间戳数据：'/home/ganlinyong/data/dataset_922_test_regular11.csv.csv'


In [ ]:
import pandas as pd
import numpy as np
import sys, os
from utils.earth_computation import distance_to_coastline
import warnings
from tqdm import tqdm
import multiprocessing as mp
import time
import os

# This shp file recorded the coastline data, and it comes from https://github.com/nvkelso/natural-earth-vector/tree/master/10m_physical. 
# We have included these files in folder 'data'. 
COASTLINE_PATH = "./data/ne_10m_coastline/ne_10m_coastline.shp"
COASTLINE_DATA = None

def init_worker():
    """初始化工作进程，加载海岸线数据"""
    global COASTLINE_DATA
    if COASTLINE_DATA is None:
        import geopandas as gpd
        COASTLINE_DATA = gpd.read_file(COASTLINE_PATH)
    
def calculate_distance(args):
    """计算单个点到海岸线的距离"""
    idx, lon, lat = args
    try:
        dist = distance_to_coastline(lon, lat, coastline_data=COASTLINE_DATA)
        return idx, dist
    except Exception as e:
        print(f"error when calculating ({lon}, {lat}): {e}")
        return idx, None

if __name__ == "__main__":
    
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    warnings.filterwarnings("ignore", category=UserWarning)
    
    df = pd.read_csv(output_dir)
    print(f"the dataset contains {len(df)} lines")
    
    init_worker()

    start_time = time.time()
    
    # 准备参数: (索引, 经度, 纬度)
    points = [(i, row['lon'], row['lat']) for i, row in df.iterrows()]
    
    # 确定进程数 (使用75%的CPU核心)
    num_cores = max(1, int(mp.cpu_count() * 0.75))
    print(f"使用 {num_cores} 个进程进行并行计算...")
    
    # 创建进程池
    with mp.Pool(processes=num_cores, initializer=init_worker) as pool:
        # 使用tqdm显示进度条
        results = list(tqdm(pool.imap(calculate_distance, points), total=len(points)))
    
    # 将结果添加到DataFrame
    for idx, dist in results:
        df.at[idx, "coastline_distance"] = dist
    
    end_time = time.time()
    print(f"并行计算完成! 耗时: {end_time - start_time:.2f} 秒")
    
    output_path = root / "dataset_with_coastline.csv"
    df.to_csv(output_path, index=False)
    print(f"结果已保存至: {output_path}")
    
    print(df.head())

数据集大小: 516113 行
使用 288 个进程进行并行计算...


100%|██████████| 516113/516113 [06:39<00:00, 1291.03it/s]


并行计算完成! 耗时: 442.20 秒
结果已保存至: /home/ganlinyong/data/dataset_922_test_with_coastline.csv
        mmsi                    postime       lat        lon        sog  \
0  215347000  2024-06-12 00:07:35+08:00  4.887450  98.341072  10.800000   
1  215347000  2024-06-12 00:12:35+08:00  4.878518  98.352968  10.693160   
2  215347000  2024-06-12 00:17:35+08:00  4.869422  98.365159  10.953671   
3  215347000  2024-06-12 00:22:35+08:00  4.860312  98.377375  10.800000   
4  215347000  2024-06-12 00:27:35+08:00  4.851189  98.389551  10.936938   

          cog  vessel_sub2_type  quarter leg_end_port_code  target_lat  \
0  126.400000             20508        2             MYPKE     3.00525   
1  127.000898             20508        2             MYPKE     3.00525   
2  126.811434             20508        2             MYPKE     3.00525   
3  126.700000             20508        2             MYPKE     3.00525   
4  126.941506             20508        2             MYPKE     3.00525   

   target_lon  co

In [ ]:
import pandas as pd
import numpy as np
from shapely.geometry import LineString
import multiprocessing as mp
from functools import partial
import time
import os
from tqdm import tqdm

start_time = time.time()

# reading strait data
print("loading strait information ...")
strait_df = pd.read_excel('./data/CapacityLargeModel_NKP_108.xlsx', sheet_name='shuhang_test') # [TODO] you should prepare this excel by yourself
# This sheet contains the information of Next Key Points, we provided a sample. 

def parse_wkt(wkt):
    """解析WKT格式的海峡线段"""
    try:
        # 提取坐标部分
        coords_str = wkt.split('(', 1)[1].rsplit(')', 1)[0]
        # 分割坐标点
        points = coords_str.split(',')
        # 转换为坐标元组列表
        coords = []
        for point in points:
            lon, lat = point.strip().split(' ')
            coords.append((float(lon), float(lat)))
        return LineString(coords)
    except Exception as e:
        print(f"解析WKT时出错: {e}")
        return None

strait_df['geometry'] = strait_df['节点矢量数据'].apply(parse_wkt)
print(f"已加载 {len(strait_df)} 个海峡")

# 读取船舶轨迹数据
print("正在读取轨迹数据...")
traj_df = pd.read_csv(output_path, parse_dates=['postime'])
traj_df.sort_values(['mmsi', 'postime'], inplace=True)
print(f"loaded {len(traj_df)} trajectories")

# 获取CPU核心数
num_cores = max(1, mp.cpu_count() - 1)
print(f"使用 {num_cores} 个CPU核心进行并行处理")

def process_vessel(group, strait_df):
    """处理单个船舶轨迹"""
    group = group.copy().reset_index(drop=True)
    n = len(group)
    
    # 初始化目标列（默认为最终目的港）
    group['next_code'] = group['leg_end_port_code']
    group['next_lat'] = group['target_lat']
    group['next_lon'] = group['target_lon']
    
    if n < 2:
        return group
    
    # 从后向前遍历轨迹点（倒数第二个点开始）
    current_target = None
    
    # 存储坐标列表提高性能
    lons = group['lon'].tolist()
    lats = group['lat'].tolist()
    ports = group['leg_end_port_code'].tolist()
    
    # 从倒数第二点开始向前遍历
    for i in tqdm(range(n-2, -1, -1)):
        # 1. 检查是否通过海峡
        segment = LineString([(lons[i], lats[i]), (lons[i+1], lats[i+1])])
        strait_found = False
        
        for _, strait in strait_df.iterrows():
            if strait['geometry'] is None:
                continue
                
            try:
                if segment.intersects(strait['geometry']):
                    # 发现海峡通过事件
                    current_target = {
                        'code': strait['node_code'],
                        'lat': strait['lat'],
                        'lon': strait['lon'],
                        'type': 'strait'
                    }
                    strait_found = True
                    break
            except Exception as e:
                print(f"几何错误: {e}")
                continue
        
        # 2. 检查目的港是否切换（仅当未发现海峡时）
        if not strait_found and ports[i] != ports[i+1]:
            # 发现目的港切换
            current_target = {
                'code': ports[i],
                'lat': group.at[i, 'target_lat'],
                'lon': group.at[i, 'target_lon'],
                'type': 'port'
            }
        
        # 3. 更新当前及之前所有点的目标值
        if current_target:
            # 更新当前点及之前所有点
            group.loc[:i, 'next_code'] = current_target['code']
            group.loc[:i, 'next_lat'] = current_target['lat']
            group.loc[:i, 'next_lon'] = current_target['lon']
    
    return group

# 并行处理函数
def parallel_processing(df, strait_df, num_cores):
    """并行处理所有船舶轨迹"""
    grouped = df.groupby('mmsi')
    worker = partial(process_vessel, strait_df=strait_df)
    
    with mp.Pool(processes=num_cores) as pool:
        results = []
        for i, result in enumerate(pool.imap_unordered(worker, [group for _, group in grouped])):
            results.append(result)
            # if (i + 1) % 100 == 0:
            #     print(f"已处理 {i+1}/{len(grouped)} 艘船舶")
    
    return pd.concat(results, ignore_index=True)

# 执行并行处理
print("开始处理轨迹数据...")
result_df = parallel_processing(traj_df, strait_df, num_cores)
print("轨迹数据处理完成!")

# save the result
output_file = root / 'dataset_final.csv'
result_df.to_csv(output_file, index=False)
print(f"结果已保存至: {output_file}")
print(f"文件大小: {os.path.getsize(output_file) / (1024*1024):.2f} MB")

# 计算总耗时
end_time = time.time()
total_time = end_time - start_time
print(f"处理完成! 总耗时: {total_time:.2f} 秒")
print(f"已增加目标海峡/港口信息列")

正在读取海峡数据...
已加载 154 个海峡
正在读取轨迹数据...
已加载 516113 条轨迹记录
使用 383 个CPU核心进行并行处理
开始处理轨迹数据...


100%|██████████| 130557/130557 [23:31<00:00, 92.49it/s]


轨迹数据处理完成!
结果已保存至: /home/ganlinyong/data/dataset_922_test_final253.csv
文件大小: 93.83 MB
处理完成! 总耗时: 1426.34 秒
已增加目标海峡/港口信息列
